# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from lib.agents import Agent, AgentState
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv

In [3]:
# Load environment variables
load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

True

In [4]:
from pydantic import BaseModel
from lib.parsers import PydanticOutputParser
from tavily import TavilyClient

class EvaluationReport(BaseModel):
    useful: bool
    description: str

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_API_BASE_URL"),
)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=5)
    games = []
    for meta in results.get("metadatas", [[]])[0]:
        games.append({
            "Platform": meta.get("Platform"),
            "Name": meta.get("Name"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Description": meta.get("Description"),
        })
    return games

#### Evaluate Retrieval Tool

In [6]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", base_url=os.getenv("OPENAI_API_BASE_URL"))
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}"
    )
    response = llm.invoke(prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    result = parser.parse(response)
    return result.model_dump_json()

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> str:
    """
    Searches the web for information about video games.
    args:
    - question: a question about game industry.
    """
    tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    results = tavily.search(query=question, max_results=5)
    simplified = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in results.get("results", [])
    ]
    return str(simplified)

### Agent

In [11]:
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.agents import AgentState

MODEL = "gpt-4o-mini"
BASE_URL = os.getenv("OPENAI_API_BASE_URL")
INSTRUCTIONS = (
    "You are UdaPlay, an AI Research Agent for the video game industry.\n\n"
    "When answering questions about games, follow this workflow:\n"
    "1. Use retrieve_game to search the internal vector database.\n"
    "2. Use evaluate_retrieval to judge whether the retrieved docs are sufficient.\n"
    "3. If the evaluation marks them as not useful, use game_web_search to find the answer online.\n"
    "4. Provide a clear, concise answer based on the information gathered.\n\n"
    "CITATION RULES — you MUST follow these in every response:\n"
    "- If the answer comes from the vector database, cite the source as: "
    "[DB: <Game Name>, <Platform>, <YearOfRelease>] after the relevant statement.\n"
    "- If the answer comes from a web search, cite the source as: "
    "[Web: <title> – <url>] after the relevant statement.\n"
    "- If both sources were used, cite each one where applicable.\n"
    "- Never give an answer without at least one citation."
)
TOOLS = [retrieve_game, evaluate_retrieval, game_web_search]

# ── Steps ─────────────────────────────────────────────────────────────────────

def message_prep(state: AgentState) -> AgentState:
    messages = state.get("messages", [])
    if not messages:
        messages = [SystemMessage(content=state["instructions"])]
    messages.append(UserMessage(content=state["user_query"]))
    return {"messages": messages}

def llm_node(state: AgentState) -> AgentState:
    llm = LLM(model=MODEL, tools=TOOLS, base_url=BASE_URL)
    response = llm.invoke(state["messages"])
    tool_calls = response.tool_calls or None
    total = state.get("total_tokens", 0)
    if response.token_usage:
        total += response.token_usage.total_tokens
    ai_msg = AIMessage(content=response.content, tool_calls=tool_calls)
    return {
        "messages": state["messages"] + [ai_msg],
        "current_tool_calls": tool_calls,
        "total_tokens": total,
    }

def _run_tool(tool_fn, state: AgentState) -> AgentState:
    tool_messages = []
    for call in (state["current_tool_calls"] or []):
        if call.function.name == tool_fn.name:
            args = json.loads(call.function.arguments)
            result = str(tool_fn(**args))
            tool_messages.append(ToolMessage(
                content=json.dumps(result),
                tool_call_id=call.id,
                name=tool_fn.name,
            ))
    return {
        "messages": state["messages"] + tool_messages,
        "current_tool_calls": None,
    }

def retrieve_game_node(state: AgentState) -> AgentState:
    return _run_tool(retrieve_game, state)

def evaluate_retrieval_node(state: AgentState) -> AgentState:
    return _run_tool(evaluate_retrieval, state)

def game_web_search_node(state: AgentState) -> AgentState:
    return _run_tool(game_web_search, state)

# ── Routing ────────────────────────────────────────────────────────────────────

_TOOL_NODE_MAP = {
    "retrieve_game":      "retrieve_game",
    "evaluate_retrieval": "evaluate_retrieval",
    "game_web_search":    "game_web_search",
}

def route_after_llm(state: AgentState):
    tool_calls = state.get("current_tool_calls")
    if not tool_calls:
        return "__termination__"
    return _TOOL_NODE_MAP.get(tool_calls[0].function.name, "__termination__")

# ── Build the State Machine ────────────────────────────────────────────────────

workflow = StateMachine[AgentState](AgentState)

entry           = EntryPoint[AgentState]()
prep_step       = Step[AgentState]("message_prep",        message_prep)
llm_step        = Step[AgentState]("llm",                 llm_node)
retrieve_step   = Step[AgentState]("retrieve_game",        retrieve_game_node)
evaluate_step   = Step[AgentState]("evaluate_retrieval",  evaluate_retrieval_node)
web_search_step = Step[AgentState]("game_web_search",     game_web_search_node)
termination     = Termination[AgentState]()

workflow.add_steps([entry, prep_step, llm_step, retrieve_step, evaluate_step, web_search_step, termination])

workflow.connect(entry,          prep_step)
workflow.connect(prep_step,      llm_step)
workflow.connect(llm_step,       [retrieve_step, evaluate_step, web_search_step, termination], route_after_llm)
workflow.connect(retrieve_step,  llm_step)
workflow.connect(evaluate_step,  llm_step)
workflow.connect(web_search_step, llm_step)

print(workflow)

StateMachine(schema=['user_query', 'instructions', 'messages', 'current_tool_calls', 'total_tokens'])


In [13]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "What platforms is it available on?",          # pronoun "it" relies on prior context
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
]

session_messages = []  # persisted across queries

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)

    initial_state: AgentState = {
        "user_query": query,
        "instructions": INSTRUCTIONS,
        "messages": session_messages,   # carry over history from previous turns
        "current_tool_calls": None,
        "total_tokens": 0,
    }

    run = workflow.run(initial_state)
    final_state = run.get_final_state()

    # Persist the full conversation for the next query
    session_messages = final_state["messages"]

    last_ai = next(
        (m for m in reversed(session_messages)
         if hasattr(m, "role") and m.role == "assistant" and m.content),
        None
    )
    print(f"Answer: {last_ai.content if last_ai else 'No answer'}")
    print(f"Tokens used: {final_state.get('total_tokens', 0)}")


Query: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm
[StateMachine] Terminating: __termination__
Answer: Pokémon Gold and Silver were released in 1999 for the Game Boy Color [DB: Pokémon Gold and Silver, Game Boy Color, 1999].
Tokens used: 2805

Query: What platforms is it available on?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm
[StateMachine] Executing step: game_web_search
[StateMachine] Executing step: llm
[StateMachine] Terminating: __termination__
Answer: Pokémon Gold and Silve

### (Optional) Advanced

In [10]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes